In [0]:
import mlflow
import mlflow.spark

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression ,RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [0]:
df = spark.table("teleco_churn_datalakehouse.gold.customer_churn_features")

In [0]:
df.display()

In [0]:
assembler = VectorAssembler(
    inputCols=[
        "Tenure_Months",
        "Monthly_Charges",
        "Total_Charges"
    ],
    outputCol="features",
    handleInvalid="skip"
)

df_features = assembler.transform(df)

In [0]:
train_df, test_df = df_features.randomSplit([0.8,0.2], seed=42)

In [0]:
mlflow.end_run()
with mlflow.start_run(run_name="Logistic_Regression_Model"):


    lr = LogisticRegression(
        featuresCol="features",
        labelCol="churn_label"
    )

    lr_model = lr.fit(train_df)

    lr_predictions = lr_model.transform(test_df)

    evaluator = BinaryClassificationEvaluator(
        labelCol="churn_label"
    )

    lr_auc = evaluator.evaluate(lr_predictions)

    print("Logistic Regression AUC:", lr_auc)

In [0]:
# Define the model
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(
    featuresCol="features",
    labelCol="churn_label"
)

In [0]:
# random forest classifier
mlflow.end_run()
with mlflow.start_run(run_name="Random_Forest_Model"):

    rf = RandomForestClassifier(
        featuresCol="features",
        labelCol="churn_label",
        numTrees=100
    )

    rf_model = rf.fit(train_df)

    rf_predictions = rf_model.transform(test_df)

    evaluator = BinaryClassificationEvaluator(
        labelCol="churn_label"
    )

    rf_auc = evaluator.evaluate(rf_predictions)

    print("Random Forest AUC:", rf_auc)

In [0]:
# MLflow requires a UC volume path for dfs_tmpdir
temp_path = "/Volumes/teleco_churn_datalakehouse/telecom_ml_database/models/tmp"

mlflow.spark.log_model(
    rf_model,
    "best_churn_model",
    dfs_tmpdir=temp_path
)

# If error persists, verify that the temp directory exists and is writable; this cannot be fixed by editing only this cell.

- Random forest was the better model

In [0]:
mlflow.spark.log_model(
    rf_model,
    "best_churn_model",
    dfs_tmpdir=temp_path
)

- Create a final predictions dataset

In [0]:
best_predictions = rf_model.transform(df_features)

- Extract churn probability

In [0]:
from pyspark.sql.functions import col

best_predictions = best_predictions.withColumn(
    "churn_probability",
    col("probability")[1]
)

- Save final predictions table

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType, ArrayType
from pyspark.ml.linalg import VectorUDT

# Extract the second element (probability of class 1)
extract_probability = udf(lambda v: float(v[1]) if v is not None else None, DoubleType())

# For binary classification, you can also use vectorToArray to convert to array first
from pyspark.ml.functions import vector_to_array

best_predictions = best_predictions.withColumn(
    "churn_probability", 
    vector_to_array("probability")[1]  # Direct array access after conversion
)

# Then select and save
best_predictions.select(
    "customer_key",
    "Tenure_Months", 
    "Monthly_Charges",
    "Total_Charges",
    "churn_probability",
    "prediction"
).write.format("delta").mode("overwrite").saveAsTable(
    "teleco_churn_datalakehouse.telecom_ml_database.churn_predictions_final"
)

- Create risk segments

In [0]:
from pyspark.sql.functions import when

risk_segments = best_predictions.withColumn(
    "risk_segment",
    when(col("churn_probability") > 0.7, "High Risk")
    .when(col("churn_probability") > 0.4, "Medium Risk")
    .otherwise("Low Risk")
)

In [0]:
risk_segments.write.format("delta").mode("overwrite").saveAsTable(
    "teleco_churn_datalakehouse.telecom_ml_database.customer_risk_segments"
)